# Agents in LangChain

## Environment Setup

### Install dependencies

#### A. on Colab?

In [ ]:
%pip install -q langchain langchain_openai langchain_community

#### B. Locally?

If using `uv` locally you can `uv sync` (provided you have the `pyproject.toml`).

### Environment variables

The following environment variables are used in this notebook:

| Variable               | Required | Purpose                                  |
|------------------------|----------|------------------------------------------|
| `OPENROUTER_API_KEY`   | Yes      | Authenticates requests to OpenRouter LLMs |

#### On Colab?

Store these as Colab **Secrets** (key icon in the left sidebar), using the variable names above (for example `OPENROUTER_API_KEY`). Grant this notebook access to the secrets so they can be read securely at runtime.

#### Locally?

Store these in a `.env` file in your project directory (which should be listed in `.gitignore` so it is never committed). The code below uses `python-dotenv` to load values from this file into environment variables at runtime.


::: {.callout-warning}

API Keys are secrets and thus [**shall never be exposed**](./never_expose_api_keys.qmd).

:::

### Reading environment variables in

In [ ]:
import os

In [ ]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

## Tools

### Web search via Tavily API (free quota)

[**Tavily**](https://www.tavily.com/) is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier.

In [2]:
%pip install tavily-python

* Tavily Search API is a search engine optimized for LLMs and RAG, aimed at efficient,
quick, and persistent search results.
* You can sign up for an API key [here](https://tavily.com/).
It's easy to sign up and offers a very generous free tier. Some lessons (in Module 4) will use Tavily.

* Add `TAVILY_API_KEY` to Colab Secrets (key icon in the left sidebar).

In [3]:
from google.colab import userdata
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=userdata.get("TAVILY_API_KEY"))

## Define `internet_search` tool

In [4]:
from typing import Literal
from langchain.tools import tool

@tool
def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

### Run the search function

In [5]:
question = "What's the timeline of how the AI industry has evolved over the past 10 years?"
result = internet_search.invoke(question, max_results=3)
result

{'query': "What's the timeline of how the AI industry has evolved over the past 10 years?",
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence',
   'title': 'Timeline of artificial intelligence - Wikipedia',
   'content': '*   [(Top)](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#). *   [3.2 1950s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1950s). *   [3.3 1960s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1960s). *   [3.4 1970s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1970s). *   [3.5 1980s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1980s). *   [3.6 1990s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1990s). *   [4.1 2000s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#2000s). *   [4.2 2010s](https://en.wikipedia.org/wiki/Timeline_of

## Create Agent (with tools)

The [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) factory function provides a production-ready **agent implementation**. It creates an agent graph that calls tools in a loop until a stopping condition is met.

[View source code on GitHub](https://github.com/langchain-ai/langchain/blob/fb6ab993a73180538f6cca876b3c85d46c08845f/libs/langchain_v1/langchain/agents/factory.py#L691).

### 1. Model: the non-deterministic program

Since agents require [**a model that supports tool calling**](https://openrouter.ai/models?fmt=cards&supported_parameters=tools). Filtering **Tool Calling** models that are also **Free**, we select NVIDIA's Nemotron-3:

In [10]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

### 2. System prompt: natural language instructions

- LLMs were tuned to prioritize **System prompt** over user messages.
    - **prompt injection attack** happens when they don't.
- LLMs have been trained on many styles of instruction to do various text output generation.

In [20]:
from textwrap import dedent


AGENT_PROMPT = dedent("""
    Role: You are an expert researcher. Your job is to conduct thorough research and then write a polished report.
    Style: Keep it short and concise.
    Tools: You have access to an `internet_search` tool as your primary means of gathering information.
    Context: the date now is 2026-04-30

    ## Tool: `internet_search`

    - Usage: Use this to run an internet search for a given query.
    - Parameters: You can specify the max number of results to return, the topic, and whether raw content should be included.
""")

- Notice that without adding `Context: the date now is 2026-04-30` the model thinks it is in `2024`! (which is the last training cut-off point).
- We might want to make this a tool-call, instead of inserting it into the prompt hard-coded like this.

See [Prompt engineering concepts](https://docs.langchain.com/langsmith/prompt-engineering-concepts) for more details.

### 3. The `agent` class instance

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model_nemotron3_nano, # Program (non-deterministic)
    tools=[internet_search],    # I/O (deterministic)
    system_prompt=AGENT_PROMPT  # Instructions (natural language)
)

In [ ]:
agent

## Running the `Agent`

- Case 1: a question that can be answered without using `internet_search`.
- Case 2: a question that needs grounding via `internet_search`.

In [13]:
from langchain.messages import HumanMessage

question = "What are the four seasons?"
steps = []
for step in agent.stream(
    {"messages": [HumanMessage(question)]},
    stream_mode="values",
):
    steps.append(step)
    step["messages"][-1].pretty_print()
result = steps[-1]

In [15]:
from rich import print

print(result)

{
    'messages': [
        HumanMessage(
            content='What are the four seasons?',
            additional_kwargs={},
            response_metadata={},
            id='03f82589-1be2-46ac-b776-1c2437297a4a'
        ),
        AIMessage(
            content='The four seasons are **spring, summer, autumn (fall), and winter**. Each season marks distinct
changes in weather, daylight, and ecological activity throughout the year.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 165,
                    'prompt_tokens': 471,
                    'total_tokens': 636,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 150,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0,
                        'upstream_inference_prompt_cost': 0,
                        'upstream_inference_completions_cost': 0
                    }
                },
                'model_provider': 'openai',
                'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free',
                'system_fingerprint': None,
                'id': 'gen-1777535330-nafjXuOIZQRmSF1vaWeK',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019ddd5c-a933-76d2-a4df-b03942d7ed18-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 471,
                'output_tokens': 165,
                'total_tokens': 636,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 150}
            }
        )
    ]
}

In [16]:
# Each streaming step pretty_prints the latest message (including the final reply).

The four seasons are **spring, summer, autumn (fall), and winter**. Each season marks distinct changes in weather, 
daylight, and ecological activity throughout the year.

Let's ask about something that needs search:

In [22]:
question = "What's the timeline of how the AI industry has evolved over the past 10 years?"
steps = []
for step in agent.stream(
    {"messages": [HumanMessage(question)]},
    stream_mode="values",
):
    steps.append(step)
    step["messages"][-1].pretty_print()
result = steps[-1]

In [23]:
print(result)

{
    'messages': [
        HumanMessage(
            content="What's the timeline of how the AI industry has evolved over the past 10 years?",
            additional_kwargs={},
            response_metadata={},
            id='0ecf6806-e0b6-42af-ac42-00f9e2344b67'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 429,
                    'prompt_tokens': 502,
                    'total_tokens': 931,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 311,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0,
                        'upstream_inference_prompt_cost': 0,
                        'upstream_inference_completions_cost': 0
                    }
                },
                'model_provider': 'openai',
                'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free',
                'system_fingerprint': None,
                'id': 'gen-1777535650-9RD4JmYBWc8Dzk7bohcH',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ddd61-8bdd-7ba1-b68a-c868f5b7c283-0',
            tool_calls=[
                {
                    'name': 'internet_search',
                    'args': {
                        'max_results': 5,
                        'topic': 'general',
                        'query': 'AI industry evolution timeline 2016 2026'
                    },
                    'id': 'call_1511273933444f6cac554dba',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 502,
                'output_tokens': 429,
                'total_tokens': 931,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 311}
            }
        ),
        ToolMessage(
            content='{"query": "AI industry evolution timeline 2016 2026", "follow_up_questions": null, "answer": 
null, "images": [], "results": [{"url": "https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence", 
"title": "Timeline of artificial intelligence - Wikipedia", "content": "*   
[(Top)](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#). *   [3.2 
1950s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1950s). *   [3.3 
1960s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1960s). *   [3.4 
1970s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1970s). *   [3.5 
1980s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1980s). *   [3.6 
1990s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#1990s). *   [4.1 
2000s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#2000s). *   [4.2 
2010s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#2010s). *   [4.3 
2020s](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#2020s). *   [5 See 
also](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#See_also). *   [9 Further 
reading](https://en.wikipedia.org/wiki/Timeline_of_artificial_intelligence#Further_reading). *   
[Read](https://en.wikipedia.org/wiki/Ti

In [24]:
# Each streaming step pretty_prints the latest message (including the final reply).

**AIIndustry Evolution (2016‑2026) – Concise Timeline**

| Year | Milestone | Impact |
|------|-----------|--------|
| **2016** | *AlphaGo* (DeepMind) defeats Lee Sedol; reinforcement‑learning breakthroughs. | First high‑profile 
demonstration of AI surpassing human experts in complex tasks. |
| **2017** | *AlphaGo Zero* and *OpenAI Dota 2* bots; BERT language model released. | Rapid progress in 
self‑learning algorithms and contextual language understanding. |
| **2018** | GPT‑1 introduced; AI research community expands with transformer architectures. | Foundations for 
today’s large‑scale language models. |
| **2019** | GPT‑2 (1.5 B parameters) released; AI begins to be integrated into consumer products. | Scaling of 
generative text models; early commercial AI services. |
| **2020** | GPT‑3 (175 B parameters) launched; AI used for drug discovery, remote work tools during COVID‑19. | 
Massive model scale drives new capabilities across industries. |
| **2021‑2022** | DALL‑E 2, Stable Diffusion, and other multimodal generators appear; EU AI Act proposal; Meta’s 
Llama series released. | Generative art and multimodal AI become mainstream; regulatory dialogue begins. |
| **2023** | GPT‑4 and multi‑modal models (text + image) released; EU AI Act adopted (phased implementation 
2024‑2027). | Consolidation of foundation models; first binding AI regulations. |
| **2024** | EU AI Act enters force; stricter compliance requirements for high‑risk AI; AI governance frameworks 
mature. | Industry shifts toward responsible AI deployment and standards. |
| **2025** | 2 M‑token context windows (e.g., Gemini 2.0 Pro); AI‑driven product‑design tools (generative design, 
digital twins) become common; AI Index and evaluation benchmarks (MLPerf, LMSYS) widely adopted. | AI moves from 
experimentation to production‑scale, enabling real‑time personalization and autonomous decisioning. |
| **2026** | Forecasts (LinkedIn, McKinsey) predict: <br>• Generative AI embedded in CX/UX design, product 
development, and marketing. <br>• Real‑time personalization and autonomous decisioning mainstream. <br>• Leaders 
exploit multi‑modal, self‑learning systems; laggards still on basic predictive analytics. | The industry reaches a 
“AI‑first” era where AI is a core driver of innovation and competitive advantage. |

**Key Takeaways**

- **2016‑2020:** Breakthroughs in deep learning and massive language models set the technical foundation.  
- **2021‑2023:** Generative and multimodal AI emerge; regulatory frameworks (EU AI Act) begin to shape the 
landscape.  
- **2024‑2026:** AI becomes embedded in everyday products and services, driven by advanced analytics, real‑time 
personalization, and autonomous systems, while compliance and standards mature.  

*Sources: Wikipedia timeline of AI, LinkedIn “AI Transformation 2026” predictions, Big Human AI history blog, 
Demicco AI timeline, Timspark AI evolution infographic.*

## Your Turn

Add the following tool to the mix, such that the agent can now what time is it, without us having to hard-code it into it's system prompt.

In [ ]:
from datetime import datetime

def get_current_datetime() -> str:
    """Return the current date and time in ISO 8601 format.
    Use this when the user refers to a date or time in a relative manner.
    Examples:
    - "in 2 days?
    - "next thursday?"
    - "next week at 2pm?"
    """
    return datetime.now().isoformat()

In [ ]:
# YOUR CODE HERE
# Step 1. Add the get_current_datetime tool to the agent
# Step 2. Make up a question that requires a reference to the current time/date or both.
# Step 3. Run the agent and see if it can answer the question correctly.

## Key Takeaways

- Agent = Model + Tools
- Models (LLMs) are the brain-power of agents
- Tools connect the LLM to the outside world.